In [1]:
from deeprobust.graph.data import Dataset, Dpr2Pyg, Pyg2Dpr
import torch
from torch_geometric.data import Data
import numpy as np
from deeprobust.graph.data import Dataset, PrePtbDataset, PtbDataset
from deeprobust.graph.defense import GCN, RGCN, ProGNN, SimPGCN, GCNSVD, GCNJaccard, GAT
from deeprobust.graph.global_attack import Metattack, DICE, Random, PGDAttack
from scipy.sparse import csr_matrix
import torch
from torch.nn import Linear
import torch.nn.functional as F
from torch_geometric.transforms import Compose
from torch_geometric.datasets import Amazon
from torch_geometric.transforms.random_node_split import RandomNodeSplit
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import NormalizeFeatures
from torch_geometric.nn import GCNConv
from torch_geometric.nn import GATConv
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import SAGEConv
from sklearn.metrics import roc_auc_score

from torch_geometric.utils import negative_sampling
from torch_geometric.utils import train_test_split_edges

from copy import deepcopy
import torch.nn as nn
from IPython.display import Javascript  # Restrict height of output cell.

In [2]:
seed = 15
ptb_rate = 0.25
# ptb_rate1 = 0.15
dataset = 'cora'
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)

In [3]:
data = Dataset(root='/tmp/', name=dataset, setting='prognn')
adj, features, labels = data.adj, data.features, data.labels
idx_train, idx_val, idx_test = data.idx_train, data.idx_val, data.idx_test
idx_unlabeled = np.union1d(idx_val, idx_test)
idx_unlabeled = np.union1d(idx_val, idx_test)


device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
budget = int(ptb_rate * (adj.todense().sum() // 2))
print(budget)
# budget1 = int(ptb_rate1 * (adj.todense().sum() // 2))
# print(budget1)

Loading cora dataset...
Selecting 1 largest connected components
1267


In [4]:
def find_low_degree_nodes(adj_matrix, k) -> np.ndarray:
    """
    Find the indices of the k nodes with the lowest degree in the graph.
    
    Parameters:
    - adj_matrix: csr_matrix, the adjacency matrix of the graph in CSR format.
    - k: int, the number of nodes to select.
    
    Returns:
    - np.ndarray, indices of the k nodes with the lowest degree.
    """
    # Compute the degree of each node. For a csr_matrix, the sum over axis=1 gives the degree.
    degrees = np.array(adj_matrix.sum(axis=1)).flatten()
    mean = np.mean(adj_matrix.sum(axis=1))
    # Find the indices of the k nodes with the lowest degrees.
    # np.argsort returns indices that would sort the array, and we take the first k elements for the lowest degrees.
    lowest_degree_nodes = np.argsort(degrees)[:k]
    
    return lowest_degree_nodes

In [5]:
import torch

def tensor_intersection(tensor1, tensor2):
    # Ensure tensors are 1D and contain unique elements
    tensor1_unique = torch.unique(tensor1)
    tensor2_unique = torch.unique(tensor2)
    
    # Sort the unique tensors for binary search
    tensor2_sorted, indices = torch.sort(tensor2_unique)
    
    # Find common elements
    mask = torch.isin(tensor1_unique, tensor2_sorted)
    intersection = tensor1_unique[mask]
    
    return intersection

In [6]:
low_degree_mask = torch.where(torch.sum(torch.tensor(adj.todense()),axis=1)>torch.mean(torch.sum(torch.tensor(adj.todense()),axis=1)))[0]

In [7]:
low_degree_nodes =tensor_intersection(low_degree_mask, torch.tensor(idx_test))

In [8]:
low_degree_nodes.shape

torch.Size([545])

In [9]:
def calculate_homophily(adj_matrix,labels, k: int) -> np.ndarray:
    n_nodes = adj_matrix.shape[0]
    degrees = np.array(adj_matrix.sum(axis=1))#.flatten()
    homophily = np.zeros(n_nodes)
    print(degrees.shape)
    # For each node, calculate the homophily.
    for i in range(n_nodes):
        #print(i)
        if degrees[i] > 0:  # To avoid division by zero
            # Find the neighbors of node i
            neighbors = adj_matrix[i].nonzero()[0]
            # Count how many of these neighbors have the same label as node i
            same_class_edges = sum(labels[neighbors] == labels[i])
            # Calculate homophily
            homophily[i] = same_class_edges / degrees[i]
        else:
            homophily[i] = 0  # Nodes with no connections have a homophily of 0 by definition

    # Find the indices of the k nodes with the lowest homophily.
    lowest_homophily_nodes = np.argsort(homophily)[:k]
    
    return lowest_homophily_nodes

# Example usage:
# Assuming adj_matrix is your scipy.sparse adjacency matrix, labels is your numpy array of labels, and you want the 5 nodes with the lowest homophily.
# k_nodes = calculate_homophily(adj_matrix, labels, 5)
# print(k_nodes)

In [10]:
low_homophily_nodes = calculate_homophily(adj.todense(),labels,1000)

(2485, 1)


In [11]:
low_homophily_nodes.shape

(1000,)

In [12]:
low_homophily_nodes=tensor_intersection(torch.tensor(low_homophily_nodes), torch.tensor(idx_test))

In [13]:
low_homophily_nodes.shape

torch.Size([805])

In [14]:
import networkx as nx

In [15]:
G=nx.from_numpy_array(adj.todense())
G.number_of_edges()

5069

In [16]:
centrality = nx.betweenness_centrality(G)

In [17]:
top_percentage = 40  # For example, top 20%

# Calculate the number of top nodes based on the percentage
num_top_nodes = int(len(centrality) * (top_percentage / 100))

# Sort the dictionary by centrality and get the top X% of node IDs
top_node_ids = sorted(centrality, key=centrality.get, reverse=True)[:num_top_nodes]

In [18]:
len(top_node_ids)

994

In [19]:
centrality_test_nodes = tensor_intersection(torch.tensor(top_node_ids), torch.tensor(idx_test))

In [20]:
centrality_test_nodes.shape

torch.Size([804])

In [21]:
idx_test.shape

(1988,)

In [22]:
features

<2485x1433 sparse matrix of type '<class 'numpy.float32'>'
	with 45487 stored elements in Compressed Sparse Row format>

In [23]:
np.save('data/cora_adj.npy',adj.todense())

In [24]:
np.save('data/cora_features.npy',features.todense())

In [25]:
np.save('data/cora_labels.npy',labels)

In [26]:
np.save('data/cora_idx_test',idx_test)

# Metattack

## GCN

In [27]:
# Setup Surrogate model
surrogate = GCN(nfeat=features.shape[1], nclass=labels.max().item()+1,
                nhid=64, dropout=0, with_relu=True, with_bias=True, device=device).to(device)
surrogate.fit(features, adj, labels, idx_train, idx_val, patience=100, verbose=True)

=== training gcn model ===
Epoch 0, training loss: 1.925281286239624
Epoch 10, training loss: 0.1152198389172554
Epoch 20, training loss: 0.014192800968885422
Epoch 30, training loss: 0.007240526378154755
Epoch 40, training loss: 0.009415091015398502
Epoch 50, training loss: 0.014282917603850365
Epoch 60, training loss: 0.016860906034708023
Epoch 70, training loss: 0.015823930501937866
Epoch 80, training loss: 0.014352404512465
Epoch 90, training loss: 0.013613451272249222
Epoch 100, training loss: 0.013121210969984531
Epoch 110, training loss: 0.012615732848644257
=== early stopping at 111, loss_val = 0.47191059589385986 ===


In [28]:
preds=surrogate.predict(features,adj)
_, predicted_classes = torch.max(preds, 1)  # Returns values, indices; we need indices

# Step 3: Subset for test nodes
test_preds = predicted_classes[idx_test].cpu().numpy()
test_labels = labels[idx_test]

# Step 4: Calculate accuracy
correct = (test_preds == test_labels).sum().item()  # Count correct predictions
total = idx_test.shape[0]  # Total number of test nodes
test_accuracy = correct / total

print(f'Test Accuracy: {test_accuracy}')

Test Accuracy: 0.8475855130784709


In [29]:
benchmark_clean = test_accuracy

In [30]:
benchmark_low = surrogate.test(low_degree_nodes)
benchmark_homo = surrogate.test(low_homophily_nodes)
benchmark_central = surrogate.test(centrality_test_nodes)

Test set results: loss= 0.4677 accuracy= 0.8624
Test set results: loss= 0.5339 accuracy= 0.8385
Test set results: loss= 0.4981 accuracy= 0.8495


In [31]:
# Setup Attack Model
model = Metattack(surrogate, nnodes=adj.shape[0], feature_shape=features.shape,
        attack_structure=True, attack_features=False, device=device, lambda_=0).to(device)
# Attack
model.attack(features, adj, labels, idx_train, idx_unlabeled, n_perturbations=budget, ll_constraint=False)
modified_adj = model.modified_adj # modified_adj is a torch.tensor
modified_adj = modified_adj.cpu().numpy()
modified_adj = csr_matrix(modified_adj)

Perturbing graph:   0%|          | 0/1267 [00:00<?, ?it/s]

GCN loss on unlabled data: 0.5024697780609131
GCN acc on unlabled data: 0.8408582923558338
attack loss: 0.24799476563930511


Perturbing graph:   0%|          | 1/1267 [00:00<10:14,  2.06it/s]

GCN loss on unlabled data: 0.5000890493392944
GCN acc on unlabled data: 0.8381761287438534
attack loss: 0.24788658320903778


Perturbing graph:   0%|          | 2/1267 [00:00<09:36,  2.20it/s]

...

GCN loss on unlabled data: 1.506003737449646
GCN acc on unlabled data: 0.7022798390701833
attack loss: 1.424990177154541


Perturbing graph:  24%|██▍       | 309/1267 [02:18<07:02,  2.27it/s]

GCN loss on unlabled data: 1.488291621208191
GCN acc on unlabled data: 0.6978095663835494
attack loss: 1.4125984907150269


Perturbing graph:  24%|██▍       | 310/1267 [02:18<07:08,  2.23it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 24.00 MiB (GPU 0; 31.74 GiB total capacity; 22.42 GiB already allocated; 9.62 MiB free; 22.43 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [505]:
from copy import deepcopy
perturbed_adj = deepcopy(modified_adj)

In [506]:
atk_model = GCN(nfeat=features.shape[1], nclass=labels.max().item()+1,
                nhid=64, dropout=0, with_relu=True, with_bias=True, device=device).to(device)
atk_model.fit(features, modified_adj, labels, idx_train, idx_val, patience=100, verbose=True)

atk_acc = atk_model.test(idx_test)
print(atk_acc)

=== training gcn model ===
Epoch 0, training loss: 1.9784939289093018
Epoch 10, training loss: 0.47519829869270325
Epoch 20, training loss: 0.10897640883922577
Epoch 30, training loss: 0.03242256119847298
Epoch 40, training loss: 0.024283969774842262
Epoch 50, training loss: 0.030382631346583366
Epoch 60, training loss: 0.032382868230342865
Epoch 70, training loss: 0.028191126883029938
Epoch 80, training loss: 0.02445906400680542
Epoch 90, training loss: 0.022598862648010254
Epoch 100, training loss: 0.021305156871676445
=== early stopping at 107, loss_val = 1.229584813117981 ===
Test set results: loss= 1.2594 accuracy= 0.5624
0.5623742454728371


In [507]:
atk_model.eval()
print((atk_acc - benchmark_clean)*100)
atk_acc = atk_model.test(low_degree_nodes)
print((atk_acc - benchmark_low)*100)
atk_acc = atk_model.test(low_homophily_nodes)
print((atk_acc - benchmark_homo)*100)
atk_acc = atk_model.test(centrality_test_nodes)
print((atk_acc - benchmark_central)*100)

-28.521126760563376
Test set results: loss= 1.1510 accuracy= 0.6165
-24.587155963302752
Test set results: loss= 1.2782 accuracy= 0.5615
-27.701863354037272
Test set results: loss= 1.1925 accuracy= 0.5833
-26.61691542288557


## GCNJaccard

In [508]:
surrogate1 = GCNJaccard(nfeat=features.shape[1],
          nhid=64,
          nclass=labels.max().item() + 1,
          dropout=0.5, device=device).to(device)
surrogate1.fit(features, adj, labels, idx_train, idx_val, threshold=0.03)

removed 1015 edges in the original graph
=== training gcn model ===
Epoch 0, training loss: 2.0463578701019287
Epoch 10, training loss: 0.21005240082740784
Epoch 20, training loss: 0.03684027120471001
Epoch 30, training loss: 0.01928817853331566
Epoch 40, training loss: 0.01740107871592045
Epoch 50, training loss: 0.023828623816370964
Epoch 60, training loss: 0.02300385944545269
Epoch 70, training loss: 0.02079833671450615
Epoch 80, training loss: 0.018904002383351326
Epoch 90, training loss: 0.015909403562545776
Epoch 100, training loss: 0.020313823595643044
Epoch 110, training loss: 0.016045548021793365
Epoch 120, training loss: 0.019576910883188248
Epoch 130, training loss: 0.01779705099761486
Epoch 140, training loss: 0.015820462256669998
Epoch 150, training loss: 0.016386542469263077
Epoch 160, training loss: 0.01549096591770649
Epoch 170, training loss: 0.013201819732785225
Epoch 180, training loss: 0.013607262633740902
Epoch 190, training loss: 0.014115337282419205
=== picking t

In [509]:
preds=surrogate1.predict(features,adj)
_, predicted_classes = torch.max(preds, 1)  # Returns values, indices; we need indices

# Step 3: Subset for test nodes
test_preds = predicted_classes[idx_test].cpu().numpy()
test_labels = labels[idx_test]

# Step 4: Calculate accuracy
correct = (test_preds == test_labels).sum().item()  # Count correct predictions
total = idx_test.shape[0]  # Total number of test nodes
test_accuracy = correct / total

print(f'Test Accuracy: {test_accuracy}')

removed 1015 edges in the original graph
Test Accuracy: 0.8148893360160966


In [510]:
# Setup Attack Model
model = Metattack(surrogate1, nnodes=adj.shape[0], feature_shape=features.shape,
        attack_structure=True, attack_features=False, device=device, lambda_=0).to(device)
# Attack
model.attack(features, adj, labels, idx_train, idx_unlabeled, n_perturbations=budget, ll_constraint=False)
modified_adj = model.modified_adj # modified_adj is a torch.tensor
modified_adj = modified_adj.cpu().numpy()
modified_adj = csr_matrix(modified_adj)

Perturbing graph:   0%|          | 0/1267 [00:00<?, ?it/s]

GCN loss on unlabled data: 0.49460217356681824
GCN acc on unlabled data: 0.83772910147519
attack loss: 0.3571441173553467


Perturbing graph:   0%|          | 1/1267 [00:00<08:46,  2.41it/s]

GCN loss on unlabled data: 0.49564146995544434
GCN acc on unlabled data: 0.8453285650424676
attack loss: 0.3619970977306366


Perturbing graph:   0%|          | 2/1267 [00:00<08:28,  2.49it/s]

...

Perturbing graph: 100%|█████████▉| 1265/1267 [07:59<00:00,  2.64it/s]

GCN loss on unlabled data: 4.780736923217773
GCN acc on unlabled data: 0.38265534197586054
attack loss: 4.9556379318237305


Perturbing graph: 100%|█████████▉| 1266/1267 [08:00<00:00,  2.63it/s]

GCN loss on unlabled data: 4.629982948303223
GCN acc on unlabled data: 0.38936075100581136
attack loss: 4.793013572692871


Perturbing graph: 100%|██████████| 1267/1267 [08:00<00:00,  2.64it/s]


In [511]:
atk_model = GCNJaccard(nfeat=features.shape[1],
          nhid=64,
          nclass=labels.max().item() + 1,
          dropout=0.5, device=device).to(device)
atk_model.fit(features, modified_adj, labels, idx_train, idx_val, patience=100, verbose=True)

atk_acc = atk_model.test(idx_test)
print(atk_acc)

removed 1042 edges in the original graph
=== training gcn model ===
Epoch 0, training loss: 1.9835352897644043
Epoch 10, training loss: 0.5467766523361206
Epoch 20, training loss: 0.18575479090213776
Epoch 30, training loss: 0.07009099423885345
Epoch 40, training loss: 0.04780782014131546
Epoch 50, training loss: 0.0440056249499321
Epoch 60, training loss: 0.04637421295046806
Epoch 70, training loss: 0.044648293405771255
Epoch 80, training loss: 0.045593347400426865
Epoch 90, training loss: 0.039841268211603165
Epoch 100, training loss: 0.03255585581064224
Epoch 110, training loss: 0.042687978595495224
Epoch 120, training loss: 0.04358834773302078
Epoch 130, training loss: 0.03351130709052086
Epoch 140, training loss: 0.03504490852355957
Epoch 150, training loss: 0.026444019749760628
Epoch 160, training loss: 0.036992959678173065
Epoch 170, training loss: 0.028210125863552094
Epoch 180, training loss: 0.037700165063142776
Epoch 190, training loss: 0.02789558470249176
=== picking the be

In [512]:
print((atk_acc - benchmark_clean)*100)

-21.730382293762574


In [513]:
atk_model.eval()
print((atk_acc - benchmark_clean)*100)
atk_acc = atk_model.test(low_degree_nodes)
print((atk_acc - benchmark_low)*100)
atk_acc = atk_model.test(low_homophily_nodes)
print((atk_acc - benchmark_homo)*100)
atk_acc = atk_model.test(centrality_test_nodes)
print((atk_acc - benchmark_central)*100)

-21.730382293762574
Test set results: loss= 0.9119 accuracy= 0.6844
-17.798165137614685
Test set results: loss= 1.0682 accuracy= 0.6211
-21.739130434782606
Test set results: loss= 0.9943 accuracy= 0.6493
-20.024875621890548


## GNN SVD

In [514]:
surrogate2 = GCNSVD(nfeat=features.shape[1],
          nhid=64,
          nclass=labels.max().item() + 1,
          dropout=0.5, device=device).to(device)
surrogate2.fit(features, adj, labels, idx_train, idx_val, k=50)

=== GCN-SVD: rank=50 ===
rank_after = 50
=== training gcn model ===
Epoch 0, training loss: 2.0478312969207764
Epoch 10, training loss: 0.4441106617450714
Epoch 20, training loss: 0.23192276060581207
Epoch 30, training loss: 0.152389258146286
Epoch 40, training loss: 0.1330999732017517
Epoch 50, training loss: 0.12049933522939682
Epoch 60, training loss: 0.10718682408332825
Epoch 70, training loss: 0.10438995063304901
Epoch 80, training loss: 0.09225352108478546
Epoch 90, training loss: 0.08834265917539597
Epoch 100, training loss: 0.09273234754800797
Epoch 110, training loss: 0.08622045069932938
Epoch 120, training loss: 0.07715422660112381
Epoch 130, training loss: 0.06571885198354721
Epoch 140, training loss: 0.07607051730155945
Epoch 150, training loss: 0.07235206663608551
Epoch 160, training loss: 0.06902556121349335
Epoch 170, training loss: 0.07407540082931519
Epoch 180, training loss: 0.059484727680683136
Epoch 190, training loss: 0.06792151182889938
=== picking the best model 

In [515]:
preds=surrogate2.predict(features,adj)
_, predicted_classes = torch.max(preds, 1)  # Returns values, indices; we need indices

# Step 3: Subset for test nodes
test_preds = predicted_classes[idx_test].cpu().numpy()
test_labels = labels[idx_test]

# Step 4: Calculate accuracy
correct = (test_preds == test_labels).sum().item()  # Count correct predictions
total = idx_test.shape[0]  # Total number of test nodes
test_accuracy = correct / total

print(f'Test Accuracy: {test_accuracy}')

=== GCN-SVD: rank=50 ===
rank_after = 50
Test Accuracy: 0.778672032193159


In [516]:
# Setup Attack Model
model = Metattack(surrogate2, nnodes=adj.shape[0], feature_shape=features.shape,
        attack_structure=True, attack_features=False, device=device, lambda_=0).to(device)
# Attack
model.attack(features, adj, labels, idx_train, idx_unlabeled, n_perturbations=budget, ll_constraint=False)
modified_adj = model.modified_adj # modified_adj is a torch.tensor
modified_adj = modified_adj.cpu().numpy()
modified_adj = csr_matrix(modified_adj)

Perturbing graph:   0%|          | 0/1267 [00:00<?, ?it/s]

GCN loss on unlabled data: 0.4906485080718994
GCN acc on unlabled data: 0.8404112650871703
attack loss: 0.7263880968093872


Perturbing graph:   0%|          | 1/1267 [00:00<08:15,  2.56it/s]

GCN loss on unlabled data: 0.4879508316516876
GCN acc on unlabled data: 0.8381761287438534
attack loss: 0.7170512676239014


Perturbing graph:   0%|          | 2/1267 [00:00<08:09,  2.58it/s]

...

Perturbing graph: 100%|█████████▉| 1265/1267 [07:58<00:00,  2.65it/s]

GCN loss on unlabled data: 3.7772648334503174
GCN acc on unlabled data: 0.5011175681716585
attack loss: 4.574840068817139


Perturbing graph: 100%|█████████▉| 1266/1267 [07:59<00:00,  2.64it/s]

GCN loss on unlabled data: 3.8277063369750977
GCN acc on unlabled data: 0.5015645954403218
attack loss: 4.631929874420166


Perturbing graph: 100%|██████████| 1267/1267 [07:59<00:00,  2.64it/s]


In [517]:
modified_adj

<2485x2485 sparse matrix of type '<class 'numpy.float32'>'
	with 12336 stored elements in Compressed Sparse Row format>

In [518]:
adj

<2485x2485 sparse matrix of type '<class 'numpy.float32'>'
	with 10138 stored elements in Compressed Sparse Row format>

In [519]:
atk_model = GCNSVD(nfeat=features.shape[1],
          nhid=64,
          nclass=labels.max().item() + 1,
          dropout=0.5, device=device).to(device)
atk_model.fit(features, modified_adj, labels, idx_train, idx_val, patience=100, verbose=True)

atk_acc = atk_model.test(idx_test)
print(atk_acc)

=== GCN-SVD: rank=50 ===
rank_after = 50
=== training gcn model ===
Epoch 0, training loss: 2.063124418258667
Epoch 10, training loss: 1.0503249168395996
Epoch 20, training loss: 0.7571268081665039
Epoch 30, training loss: 0.6466550827026367
Epoch 40, training loss: 0.5828679800033569
Epoch 50, training loss: 0.5121331810951233
Epoch 60, training loss: 0.49745988845825195
Epoch 70, training loss: 0.4521006941795349
Epoch 80, training loss: 0.43865278363227844
Epoch 90, training loss: 0.4213291108608246
Epoch 100, training loss: 0.41996997594833374
Epoch 110, training loss: 0.3958130180835724
Epoch 120, training loss: 0.3695931136608124
Epoch 130, training loss: 0.36434823274612427
Epoch 140, training loss: 0.36248210072517395
Epoch 150, training loss: 0.34426409006118774
Epoch 160, training loss: 0.34163880348205566
Epoch 170, training loss: 0.34313517808914185
Epoch 180, training loss: 0.3145664930343628
Epoch 190, training loss: 0.31738659739494324
=== picking the best model accordin

In [520]:
atk_model.eval()
print((atk_acc - benchmark_clean)*100)
atk_acc = atk_model.test(low_degree_nodes)
print((atk_acc - benchmark_low)*100)
atk_acc = atk_model.test(low_homophily_nodes)
print((atk_acc - benchmark_homo)*100)
atk_acc = atk_model.test(centrality_test_nodes)
print((atk_acc - benchmark_central)*100)

-19.215291750503017
Test set results: loss= 1.0368 accuracy= 0.6881
-17.431192660550455
Test set results: loss= 1.1420 accuracy= 0.6422
-19.627329192546583
Test set results: loss= 1.0650 accuracy= 0.6542
-19.527363184079604


## SimPGCN

In [521]:
surrogate3 = SimPGCN(nnodes=features.shape[0], nfeat=features.shape[1],
    nhid=16, nclass=labels.max()+1, device=device).to(device)

surrogate3.fit(features, adj, labels, idx_train, idx_val, train_iters=200, verbose=True)
surrogate3.eval()
test_accuracy = surrogate3.test(idx_test)
print(test_accuracy)

=== training gcn model ===
loading saved_knn/knn_graph_(2485, 1433).npz...
loading saved_knn/cosine_sims_(2485, 1433).npy
loading saved_knn/attrsim_sampled_idx_(2485, 1433).npy
number of sampled: 21982
Epoch 0, training loss: 2.0416417121887207
Epoch 10, training loss: 1.3900383710861206
Epoch 20, training loss: 0.6495118141174316
Epoch 30, training loss: 0.29332712292671204
Epoch 40, training loss: 0.10262002050876617
Epoch 50, training loss: 0.062350060790777206
Epoch 60, training loss: 0.04831228032708168
Epoch 70, training loss: 0.022217363119125366
Epoch 80, training loss: 0.028409525752067566
Epoch 90, training loss: 0.024171579629182816
Epoch 100, training loss: 0.02080116979777813
Epoch 110, training loss: 0.015873635187745094
Epoch 120, training loss: 0.018329283222556114
Epoch 130, training loss: 0.021953802555799484
Epoch 140, training loss: 0.02410258911550045
Epoch 150, training loss: 0.023288873955607414
Epoch 160, training loss: 0.016698773950338364
Epoch 170, training l

In [522]:
preds=surrogate3.predict(features,adj)
_, predicted_classes = torch.max(preds, 1)  # Returns values, indices; we need indices

# Step 3: Subset for test nodes
test_preds = predicted_classes[idx_test].cpu().numpy()
test_labels = labels[idx_test]

# Step 4: Calculate accuracy
correct = (test_preds == test_labels).sum().item()  # Count correct predictions
total = idx_test.shape[0]  # Total number of test nodes
test_accuracy = correct / total

print(f'Test Accuracy: {test_accuracy}')

Test Accuracy: 0.8234406438631791


In [523]:
# print('==================')
# print('=== load graph perturbed by DeepRobust 5% metattack (under prognn splits) ===')
# perturbed_data = PtbDataset(root='/tmp/',
#                     name=dataset,
#                     attack_method='meta')
# perturbed_adj = perturbed_data.adj

# surrogate3.fit(features, perturbed_adj, labels, idx_train, train_iters=200, verbose=True)
# surrogate3.eval()
# # You can use the inner function of model to test
# atk_acc = surrogate3.test(idx_test)
# print(atk_acc)

In [524]:
surrogate3 = SimPGCN(nnodes=features.shape[0], nfeat=features.shape[1],
    nhid=16, nclass=labels.max()+1, device=device).to(device)

surrogate3.fit(features, perturbed_adj, labels, idx_train, train_iters=200, verbose=True)
surrogate3.eval()
# You can use the inner function of model to test
atk_acc = surrogate3.test(idx_test)
print(atk_acc)

loading saved_knn/knn_graph_(2485, 1433).npz...
loading saved_knn/cosine_sims_(2485, 1433).npy
loading saved_knn/attrsim_sampled_idx_(2485, 1433).npy
number of sampled: 21982
Epoch 0, training loss: 2.0342206954956055
Epoch 10, training loss: 1.6237754821777344
Epoch 20, training loss: 1.1568529605865479
Epoch 30, training loss: 0.6382514834403992
Epoch 40, training loss: 0.3803112506866455
Epoch 50, training loss: 0.18162216246128082
Epoch 60, training loss: 0.08190419524908066
Epoch 70, training loss: 0.0686241015791893
Epoch 80, training loss: 0.03422907367348671
Epoch 90, training loss: 0.050814274698495865
Epoch 100, training loss: 0.04091096296906471
Epoch 110, training loss: 0.03279523178935051
Epoch 120, training loss: 0.038740918040275574
Epoch 130, training loss: 0.023855166509747505
Epoch 140, training loss: 0.019291114062070847
Epoch 150, training loss: 0.023487213999032974
Epoch 160, training loss: 0.013334927149116993
Epoch 170, training loss: 0.024117982015013695
Epoch 1

In [525]:
surrogate3.eval()
print((atk_acc - benchmark_clean)*100)
atk_acc = surrogate3.test(low_degree_nodes)
print((atk_acc - benchmark_low)*100)
atk_acc = surrogate3.test(low_homophily_nodes)
print((atk_acc - benchmark_homo)*100)
atk_acc = surrogate3.test(centrality_test_nodes)
print((atk_acc - benchmark_central)*100)

-38.43058350100603
Test set results: loss= 5.4220 accuracy= 0.5174
-34.4954128440367
Test set results: loss= 5.2998 accuracy= 0.4323
-40.621118012422365
Test set results: loss= 4.9893 accuracy= 0.4838
-36.56716417910448


## ProGNN

In [526]:
fit_kwargs = dict(no_cuda=False, seed=15 ,only_gcn=False,debug=False,lr=0.01,weight_decay=0.0005,hidden=64,dropout=0.5,ptb_rate=ptb_rate,epochs=100,alpha=5e-4,beta=1.5,gamma=1,
                  lambda_=0,phi=0,inner_steps=2, outer_steps=1,lr_adj=0.01, symmetric=True)

In [527]:
import argparse
args = argparse.Namespace(**fit_kwargs)


In [528]:
model = GCN(nfeat=features.shape[1],
            nhid=64,
            nclass=labels.max().item() + 1,
            dropout=0.5, device=device)

In [529]:
prognn = ProGNN(model,args,device=device)

In [530]:
features

<2485x1433 sparse matrix of type '<class 'numpy.float32'>'
	with 45487 stored elements in Compressed Sparse Row format>

In [531]:
prognn.fit(torch.tensor(features.todense()).cuda(), torch.tensor(adj.todense()).cuda(), torch.tensor(labels).long().cuda(), 
           torch.tensor(idx_train).cuda(), torch.tensor(idx_val))

Epoch: 0001 acc_train: 0.0648 loss_val: 2.1213 acc_val: 0.0482 time: 2.7795s
Epoch: 0002 acc_train: 0.8138 loss_val: 1.5991 acc_val: 0.6707 time: 3.0039s
Epoch: 0003 acc_train: 0.8178 loss_val: 1.1665 acc_val: 0.6787 time: 2.7705s
Epoch: 0004 acc_train: 0.8623 loss_val: 0.8618 acc_val: 0.7470 time: 2.8874s
Epoch: 0005 acc_train: 0.9312 loss_val: 0.6410 acc_val: 0.8353 time: 2.8847s
Epoch: 0006 acc_train: 0.9474 loss_val: 0.5236 acc_val: 0.8394 time: 2.9406s
Epoch: 0007 acc_train: 0.9555 loss_val: 0.4752 acc_val: 0.8474 time: 2.9341s
Epoch: 0008 acc_train: 0.9838 loss_val: 0.4635 acc_val: 0.8394 time: 2.6923s
Epoch: 0009 acc_train: 0.9919 loss_val: 0.4606 acc_val: 0.8313 time: 2.5872s
Epoch: 0010 acc_train: 0.9960 loss_val: 0.4615 acc_val: 0.8394 time: 2.7642s
Epoch: 0011 acc_train: 0.9960 loss_val: 0.4720 acc_val: 0.8313 time: 2.9999s
Epoch: 0012 acc_train: 0.9960 loss_val: 0.4851 acc_val: 0.8353 time: 2.7780s
Epoch: 0013 acc_train: 1.0000 loss_val: 0.4965 acc_val: 0.8394 time: 2.7152s

In [532]:
model = GCN(nfeat=features.shape[1],
            nhid=64,
            nclass=labels.max().item() + 1,
            dropout=0.5, device=device)
surrogate4 = ProGNN(model,args,device=device)
surrogate4.fit(torch.tensor(features.todense()).cuda(), torch.tensor(perturbed_adj.todense()).cuda(), torch.tensor(labels).long().cuda(), 
               torch.tensor(idx_train).cuda(), torch.tensor(idx_val))

Epoch: 0001 acc_train: 0.1700 loss_val: 1.9726 acc_val: 0.1084 time: 3.1407s
Epoch: 0002 acc_train: 0.4008 loss_val: 1.7011 acc_val: 0.3012 time: 2.9264s
Epoch: 0003 acc_train: 0.7247 loss_val: 1.5102 acc_val: 0.4418 time: 2.8829s
Epoch: 0004 acc_train: 0.7895 loss_val: 1.3805 acc_val: 0.5502 time: 2.7192s
Epoch: 0005 acc_train: 0.8623 loss_val: 1.3068 acc_val: 0.5622 time: 2.6207s
Epoch: 0006 acc_train: 0.9069 loss_val: 1.3611 acc_val: 0.5582 time: 2.9193s
Epoch: 0007 acc_train: 0.9069 loss_val: 1.4831 acc_val: 0.5382 time: 3.4102s
Epoch: 0008 acc_train: 0.9271 loss_val: 1.6246 acc_val: 0.5221 time: 2.8897s
Epoch: 0009 acc_train: 0.9433 loss_val: 1.7754 acc_val: 0.5020 time: 3.3291s
Epoch: 0010 acc_train: 0.9676 loss_val: 2.0207 acc_val: 0.4819 time: 2.9328s
Epoch: 0011 acc_train: 0.9757 loss_val: 2.2511 acc_val: 0.4659 time: 2.8597s
Epoch: 0012 acc_train: 0.9757 loss_val: 2.4060 acc_val: 0.4699 time: 2.6655s
Epoch: 0013 acc_train: 0.9838 loss_val: 2.5829 acc_val: 0.4498 time: 2.8105s

In [533]:
surrogate4.test(torch.tensor(features.todense()).cuda(),torch.tensor(labels).long().cuda(), torch.tensor(idx_test).cuda())

	=== testing ===
	Test set results: loss= 1.3562 accuracy= 0.5448


0.5447686116700202

In [534]:
perturbed_adj

<2485x2485 sparse matrix of type '<class 'numpy.float32'>'
	with 12548 stored elements in Compressed Sparse Row format>

In [535]:
#surrogate4.eval()
atk_acc=surrogate4.test(torch.tensor(features.todense()).cuda(),torch.tensor(labels).long().cuda(), torch.tensor(idx_test).cuda())
print((atk_acc - benchmark_clean)*100)
atk_acc =surrogate4.test(torch.tensor(features.todense()).cuda(),torch.tensor(labels).long().cuda(), torch.tensor(low_degree_nodes).cuda())
print((atk_acc - benchmark_low)*100)
atk_acc = surrogate4.test(torch.tensor(features.todense()).cuda(),torch.tensor(labels).long().cuda(), torch.tensor(low_homophily_nodes).cuda())
print((atk_acc - benchmark_homo)*100)
atk_acc = surrogate4.test(torch.tensor(features.todense()).cuda(),torch.tensor(labels).long().cuda(), torch.tensor(centrality_test_nodes).cuda())
print((atk_acc - benchmark_central)*100)

	=== testing ===
	Test set results: loss= 1.3562 accuracy= 0.5448
-30.281690140845065
	=== testing ===
	Test set results: loss= 1.2541 accuracy= 0.6000
-26.23853211009174
	=== testing ===
	Test set results: loss= 1.3668 accuracy= 0.5590
-27.95031055900621
	=== testing ===
	Test set results: loss= 1.2915 accuracy= 0.5771
-27.23880597014925


/tmp/ipykernel_384374/782951187.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  atk_acc =surrogate4.test(torch.tensor(features.todense()).cuda(),torch.tensor(labels).long().cuda(), torch.tensor(low_degree_nodes).cuda())
/tmp/ipykernel_384374/782951187.py:6: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  atk_acc = surrogate4.test(torch.tensor(features.todense()).cuda(),torch.tensor(labels).long().cuda(), torch.tensor(low_homophily_nodes).cuda())
/tmp/ipykernel_384374/782951187.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  atk_acc 

## GAT

In [536]:
surrogate5 = GAT(nfeat=features.shape[1],
      nhid=8, heads=8,
      nclass=labels.max().item() + 1,
      dropout=0.5, device=device)
surrogate5 = surrogate5.to(device)

In [537]:
pyg_data = Dpr2Pyg(data)
surrogate5.fit(pyg_data, verbose=True) # train with earlystopping

Processing...
Done!


=== training GAT model ===
Epoch 0, training loss: 1.956689715385437
Epoch 10, training loss: 0.7015579342842102
Epoch 20, training loss: 0.4865920841693878
Epoch 30, training loss: 0.41292354464530945
Epoch 40, training loss: 0.2921927869319916
Epoch 50, training loss: 0.35568955540657043
Epoch 60, training loss: 0.28701382875442505
Epoch 70, training loss: 0.32561472058296204
Epoch 80, training loss: 0.2591412663459778
Epoch 90, training loss: 0.2778574824333191
Epoch 100, training loss: 0.29040801525115967
Epoch 110, training loss: 0.27197933197021484
Epoch 120, training loss: 0.3072277009487152
=== early stopping at 120, loss_val = 0.44488462805747986 ===


In [538]:
surrogate5.test()

Test set results: loss= 0.4747 accuracy= 0.8405


0.8405432595573441

In [539]:
pyg_data.update_edge_index(perturbed_adj) # inplace operation

In [540]:
surrogate5 = GAT(nfeat=features.shape[1],
      nhid=8, heads=8,
      nclass=labels.max().item() + 1,
      dropout=0.5, device=device)
surrogate5 = surrogate5.to(device)

In [541]:
surrogate5.fit(pyg_data, verbose=True) # train with earlystopping

=== training GAT model ===
Epoch 0, training loss: 1.9587695598602295
Epoch 10, training loss: 0.9202792644500732
Epoch 20, training loss: 0.560576856136322
Epoch 30, training loss: 0.44026511907577515
Epoch 40, training loss: 0.4474378526210785
Epoch 50, training loss: 0.33317965269088745
Epoch 60, training loss: 0.3199295103549957
Epoch 70, training loss: 0.33402660489082336
Epoch 80, training loss: 0.30851680040359497
Epoch 90, training loss: 0.31185415387153625
Epoch 100, training loss: 0.3012942373752594
Epoch 110, training loss: 0.29589343070983887
=== early stopping at 110, loss_val = 1.1852664947509766 ===


In [542]:
surrogate5.eval()
logits = surrogate5.predict()

In [543]:
labels

array([5, 2, 0, ..., 2, 2, 2], dtype=int8)

In [544]:
import torch

def calculate_accuracy(logits, labels, test_ids):
    """
    Calculate the prediction accuracy.
    
    Parameters:
    - logits: Tensor of model output features with LogSoftmax applied.
    - labels: Tensor of ground truth labels.
    - test_ids: Tensor or list of indices for test samples.
    
    Returns:
    - accuracy: Float representing the prediction accuracy.
    """
    # Convert logits to predicted class labels
    _, predicted_labels = torch.max(logits, dim=1)
    
    # Select the predictions and labels for the test set
    test_predictions = predicted_labels[test_ids]
    test_labels = labels[test_ids]
    
    # Calculate accuracy
    correct_predictions = torch.sum(test_predictions == test_labels)
    accuracy = correct_predictions.float() / test_ids.size(0)
    
    return accuracy.item()  # Convert to Python float

# Example usage
# Assuming `logits`, `labels_tensor`, and `test_ids` are defined and available
accuracy = calculate_accuracy(logits, torch.tensor(labels).cuda(), torch.tensor(idx_test).cuda())
print(f"Test Accuracy: {(accuracy-benchmark_clean)*100}")

Test Accuracy: -27.71629829042155


In [545]:
accuracy = calculate_accuracy(logits, torch.tensor(labels).cuda(), low_degree_nodes.cuda())
print(f"Low Degree Accuracy: {(accuracy-benchmark_low)*100}")
accuracy = calculate_accuracy(logits, torch.tensor(labels).cuda(), low_homophily_nodes.cuda())
print(f"Low Homophily Accuracy: {(accuracy-benchmark_homo)*100}")
accuracy = calculate_accuracy(logits, torch.tensor(labels).cuda(), centrality_test_nodes.cuda())
print(f"Low Homophily Accuracy: {(accuracy-benchmark_central)*100}")

Low Degree Accuracy: -27.155964418288768
Low Homophily Accuracy: -29.068326505815023
Low Homophily Accuracy: -26.74129442195987


# DICE

## GCN

In [156]:
# Setup Surrogate model
surrogate = GCN(nfeat=features.shape[1], nclass=labels.max().item()+1,
                nhid=64, dropout=0, with_relu=True, with_bias=True, device=device).to(device)
surrogate.fit(features, adj, labels, idx_train, idx_val, patience=100, verbose=True)

=== training gcn model ===
Epoch 0, training loss: 2.0013506412506104
Epoch 10, training loss: 0.1784532070159912
Epoch 20, training loss: 0.02265828847885132
Epoch 30, training loss: 0.008995148353278637
Epoch 40, training loss: 0.01016619335860014
Epoch 50, training loss: 0.014811103232204914
Epoch 60, training loss: 0.01746569201350212
Epoch 70, training loss: 0.016285421326756477
Epoch 80, training loss: 0.01452041044831276
Epoch 90, training loss: 0.01350044459104538
Epoch 100, training loss: 0.012842738069593906
Epoch 110, training loss: 0.012275326065719128
=== early stopping at 113, loss_val = 0.4966380298137665 ===


In [157]:
preds=surrogate.predict(features,adj)
_, predicted_classes = torch.max(preds, 1)  # Returns values, indices; we need indices

# Step 3: Subset for test nodes
test_preds = predicted_classes[idx_test].cpu().numpy()
test_labels = labels[idx_test]

# Step 4: Calculate accuracy
correct = (test_preds == test_labels).sum().item()  # Count correct predictions
total = idx_test.shape[0]  # Total number of test nodes
test_accuracy = correct / total

print(f'Test Accuracy: {test_accuracy}')

Test Accuracy: 0.8350100603621731


In [158]:
# Setup Attack Model
model = DICE(surrogate, nnodes=adj.shape[0], attack_structure=True, attack_features=False, device=device).to(device)
model.attack(adj, labels, n_perturbations=budget)
modified_adj = model.modified_adj
# Convert SciPy sparse matrix to PyTorch tensor
modified_adj_torch = torch.tensor(modified_adj.toarray(), dtype=torch.float)

# Now you can safely call .cpu() if needed (useful if working with CUDA tensors)
modified_adj_torch = modified_adj_torch.cpu()

modified_adj = csr_matrix(modified_adj_torch)

number of pertubations: 253


In [159]:
atk_model = GCN(nfeat=features.shape[1], nclass=labels.max().item()+1,
                nhid=64, dropout=0, with_relu=True, with_bias=True, device=device).to(device)
atk_model.fit(features, modified_adj, labels, idx_train, idx_val, patience=100, verbose=True)

atk_acc = atk_model.test(idx_test)
print(atk_acc)

=== training gcn model ===
Epoch 0, training loss: 1.9988664388656616
Epoch 10, training loss: 0.1770801693201065
Epoch 20, training loss: 0.02095688320696354
Epoch 30, training loss: 0.008767915889620781
Epoch 40, training loss: 0.010151972994208336
Epoch 50, training loss: 0.01509387232363224
Epoch 60, training loss: 0.017798857763409615
Epoch 70, training loss: 0.016668790951371193
Epoch 80, training loss: 0.014997268095612526
Epoch 90, training loss: 0.01406441256403923
Epoch 100, training loss: 0.013466321863234043
Epoch 110, training loss: 0.012926824390888214
=== early stopping at 112, loss_val = 0.5503204464912415 ===
Test set results: loss= 0.5443 accuracy= 0.8234
0.8234406438631792


In [160]:
print((atk_acc - benchmark_clean)*100)

-2.4144869215291687


## GCNSVD

In [161]:
surrogate2 = GCNSVD(nfeat=features.shape[1],
          nhid=64,
          nclass=labels.max().item() + 1,
          dropout=0.5, device=device).to(device)
surrogate2.fit(features, adj, labels, idx_train, idx_val, k=50)

=== GCN-SVD: rank=50 ===
rank_after = 50
=== training gcn model ===
Epoch 0, training loss: 1.9225667715072632
Epoch 10, training loss: 0.3862052857875824
Epoch 20, training loss: 0.20534750819206238
Epoch 30, training loss: 0.14677411317825317
Epoch 40, training loss: 0.11764820665121078
Epoch 50, training loss: 0.1066364124417305
Epoch 60, training loss: 0.0935211107134819
Epoch 70, training loss: 0.09862826019525528
Epoch 80, training loss: 0.08794967085123062
Epoch 90, training loss: 0.08464223146438599
Epoch 100, training loss: 0.07630077749490738
Epoch 110, training loss: 0.06864850968122482
Epoch 120, training loss: 0.07030574232339859
Epoch 130, training loss: 0.06677495688199997
Epoch 140, training loss: 0.06452471762895584
Epoch 150, training loss: 0.06983944028615952
Epoch 160, training loss: 0.06885287910699844
Epoch 170, training loss: 0.053785767406225204
Epoch 180, training loss: 0.06173109635710716
Epoch 190, training loss: 0.07087459415197372
=== picking the best model

In [162]:
preds=surrogate2.predict(features,adj)
_, predicted_classes = torch.max(preds, 1)  # Returns values, indices; we need indices

# Step 3: Subset for test nodes
test_preds = predicted_classes[idx_test].cpu().numpy()
test_labels = labels[idx_test]

# Step 4: Calculate accuracy
correct = (test_preds == test_labels).sum().item()  # Count correct predictions
total = idx_test.shape[0]  # Total number of test nodes
test_accuracy = correct / total

print(f'Test Accuracy: {test_accuracy}')

=== GCN-SVD: rank=50 ===
rank_after = 50
Test Accuracy: 0.778672032193159


In [163]:
# Setup Attack Model
model = DICE(surrogate2, nnodes=adj.shape[0], attack_structure=True, attack_features=False, device=device).to(device)
model.attack(adj, labels, n_perturbations=budget)

modified_adj = model.modified_adj
# Convert SciPy sparse matrix to PyTorch tensor
modified_adj_torch = torch.tensor(modified_adj.toarray(), dtype=torch.float)

# Now you can safely call .cpu() if needed (useful if working with CUDA tensors)
modified_adj_torch = modified_adj_torch.cpu()

modified_adj = csr_matrix(modified_adj_torch)

number of pertubations: 253


In [164]:
atk_model = GCNSVD(nfeat=features.shape[1],
          nhid=64,
          nclass=labels.max().item() + 1,
          dropout=0.5, device=device).to(device)
atk_model.fit(features, modified_adj, labels, idx_train, idx_val, patience=100, verbose=True)

atk_acc = atk_model.test(idx_test)
print(atk_acc)

=== GCN-SVD: rank=50 ===
rank_after = 50
=== training gcn model ===
Epoch 0, training loss: 1.9886462688446045
Epoch 10, training loss: 0.4526948034763336
Epoch 20, training loss: 0.2588203251361847
Epoch 30, training loss: 0.17912545800209045
Epoch 40, training loss: 0.14748014509677887
Epoch 50, training loss: 0.14473453164100647
Epoch 60, training loss: 0.1214304119348526
Epoch 70, training loss: 0.11344599723815918
Epoch 80, training loss: 0.11132913082838058
Epoch 90, training loss: 0.09962953627109528
Epoch 100, training loss: 0.09191514551639557
Epoch 110, training loss: 0.08824841678142548
Epoch 120, training loss: 0.10258487612009048
Epoch 130, training loss: 0.07447312027215958
Epoch 140, training loss: 0.08269505202770233
Epoch 150, training loss: 0.08718709647655487
Epoch 160, training loss: 0.07425755262374878
Epoch 170, training loss: 0.0755540207028389
Epoch 180, training loss: 0.07198939472436905
Epoch 190, training loss: 0.07253514230251312
=== picking the best model a

In [165]:
print((atk_acc - benchmark_clean)*100)

-8.55130784708249


## GCNJaccard

In [166]:
surrogate1 = GCNJaccard(nfeat=features.shape[1],
          nhid=64,
          nclass=labels.max().item() + 1,
          dropout=0.5, device=device).to(device)
surrogate1.fit(features, adj, labels, idx_train, idx_val, threshold=0.03)

removed 1015 edges in the original graph
=== training gcn model ===
Epoch 0, training loss: 1.974871039390564
Epoch 10, training loss: 0.20419029891490936
Epoch 20, training loss: 0.03915862366557121
Epoch 30, training loss: 0.019494246691465378
Epoch 40, training loss: 0.015022391453385353
Epoch 50, training loss: 0.020247647538781166
Epoch 60, training loss: 0.018639886751770973
Epoch 70, training loss: 0.023240288719534874
Epoch 80, training loss: 0.019225651398301125
Epoch 90, training loss: 0.017367009073495865
Epoch 100, training loss: 0.012840601615607738
Epoch 110, training loss: 0.01236194558441639
Epoch 120, training loss: 0.017497288063168526
Epoch 130, training loss: 0.015027550049126148
Epoch 140, training loss: 0.016727082431316376
Epoch 150, training loss: 0.012636402621865273
Epoch 160, training loss: 0.016095105558633804
Epoch 170, training loss: 0.01243385300040245
Epoch 180, training loss: 0.011558675207197666
Epoch 190, training loss: 0.01197216659784317
=== picking

In [167]:
preds=surrogate1.predict(features,adj)
_, predicted_classes = torch.max(preds, 1)  # Returns values, indices; we need indices

# Step 3: Subset for test nodes
test_preds = predicted_classes[idx_test].cpu().numpy()
test_labels = labels[idx_test]

# Step 4: Calculate accuracy
correct = (test_preds == test_labels).sum().item()  # Count correct predictions
total = idx_test.shape[0]  # Total number of test nodes
test_accuracy = correct / total

print(f'Test Accuracy: {test_accuracy}')

removed 1015 edges in the original graph
Test Accuracy: 0.8204225352112676


In [168]:
# Setup Attack Model
model = DICE(surrogate1, nnodes=adj.shape[0], attack_structure=True, attack_features=False, device=device).to(device)
model.attack(adj, labels, n_perturbations=budget)

modified_adj = model.modified_adj
# Convert SciPy sparse matrix to PyTorch tensor
modified_adj_torch = torch.tensor(modified_adj.toarray(), dtype=torch.float)

# Now you can safely call .cpu() if needed (useful if working with CUDA tensors)
modified_adj_torch = modified_adj_torch.cpu()

modified_adj = csr_matrix(modified_adj_torch)

number of pertubations: 253


In [169]:
atk_model = GCNJaccard(nfeat=features.shape[1],
          nhid=64,
          nclass=labels.max().item() + 1,
          dropout=0.5, device=device).to(device)
atk_model.fit(features, modified_adj, labels, idx_train, idx_val, patience=100, verbose=True)

atk_acc = atk_model.test(idx_test)
print(atk_acc)

removed 592 edges in the original graph
=== training gcn model ===
Epoch 0, training loss: 1.9543516635894775
Epoch 10, training loss: 0.23513753712177277
Epoch 20, training loss: 0.0553223118185997
Epoch 30, training loss: 0.017869023606181145
Epoch 40, training loss: 0.01658315770328045
Epoch 50, training loss: 0.02238362282514572
Epoch 60, training loss: 0.018894152715802193
Epoch 70, training loss: 0.02389240451157093
Epoch 80, training loss: 0.017526455223560333
Epoch 90, training loss: 0.020820094272494316
Epoch 100, training loss: 0.02099476009607315
Epoch 110, training loss: 0.016860488802194595
Epoch 120, training loss: 0.015659354627132416
Epoch 130, training loss: 0.0217744130641222
Epoch 140, training loss: 0.01514825876802206
Epoch 150, training loss: 0.016387227922677994
Epoch 160, training loss: 0.01557573489844799
Epoch 170, training loss: 0.015020075254142284
Epoch 180, training loss: 0.013103967532515526
Epoch 190, training loss: 0.01770988292992115
=== picking the be

In [170]:
benchmark_clean

0.8475855130784709

In [171]:
print((atk_acc - benchmark_clean)*100)

-3.9738430583501017


## SimPGCN

In [8]:
surrogate3 = SimPGCN(nnodes=features.shape[0], nfeat=features.shape[1],
    nhid=16, nclass=labels.max()+1, device=device).to(device)

surrogate3.fit(features, adj, labels, idx_train, idx_val, train_iters=200, verbose=True)

/usr/local/lib/python3.10/dist-packages/deeprobust/graph/utils.py:356: UserWarning: torch.sparse.SparseTensor(indices, values, shape, *, device=) is deprecated.  Please use torch.sparse_coo_tensor(indices, values, shape, dtype=, device=). (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:618.)
  return torch.sparse.FloatTensor(sparseconcat.t(),sparsedata,torch.Size(sparse_mx.shape))


=== training gcn model ===
loading saved_knn/cosine_sims_(2485, 1433).npy
number of sampled: 21972
Epoch 0, training loss: 1.9035120010375977
Epoch 10, training loss: 1.3902671337127686
Epoch 20, training loss: 0.9233468174934387
Epoch 30, training loss: 0.491983562707901
Epoch 40, training loss: 0.212081179022789
Epoch 50, training loss: 0.09256990998983383
Epoch 60, training loss: 0.048811428248882294
Epoch 70, training loss: 0.03520660102367401
Epoch 80, training loss: 0.022107426077127457
Epoch 90, training loss: 0.01542553212493658
Epoch 100, training loss: 0.015348459593951702
Epoch 110, training loss: 0.03367571532726288
Epoch 120, training loss: 0.02575494721531868
Epoch 130, training loss: 0.0166032537817955
Epoch 140, training loss: 0.02154587209224701
Epoch 150, training loss: 0.01488436572253704
Epoch 160, training loss: 0.011220656335353851
Epoch 170, training loss: 0.011381352320313454
Epoch 180, training loss: 0.01340511441230774
Epoch 190, training loss: 0.0183904431760

In [10]:
preds=surrogate3.predict(features,adj)
_, predicted_classes = torch.max(preds, 1)  # Returns values, indices; we need indices

# Step 3: Subset for test nodes
test_preds = predicted_classes[idx_test].cpu().numpy()
test_labels = labels[idx_test]

# Step 4: Calculate accuracy
correct = (test_preds == test_labels).sum().item()  # Count correct predictions
total = idx_test.shape[0]  # Total number of test nodes
test_accuracy = correct / total

print(f'Test Accuracy: {test_accuracy}')

Test Accuracy: 0.8254527162977867


In [14]:
# Setup Attack Model
model = DICE(surrogate3, nnodes=adj.shape[0], attack_structure=True, attack_features=False, device=device).to(device)
model.attack(adj, labels, n_perturbations=budget)

modified_adj = model.modified_adj
# Convert SciPy sparse matrix to PyTorch tensor
modified_adj_torch = torch.tensor(modified_adj.toarray(), dtype=torch.float)

# Now you can safely call .cpu() if needed (useful if working with CUDA tensors)
modified_adj_torch = modified_adj_torch.cpu()

modified_adj = csr_matrix(modified_adj_torch)

number of pertubations: 253


In [15]:
atk_model = SimPGCN(nnodes=features.shape[0], nfeat=features.shape[1],
    nhid=16, nclass=labels.max()+1, device=device).to(device)
atk_model.fit(features, modified_adj, labels, idx_train, idx_val, patience=100, verbose=True)

atk_acc = atk_model.test(idx_test)
print(atk_acc)

=== training gcn model ===
loading saved_knn/knn_graph_(2485, 1433).npz...
loading saved_knn/cosine_sims_(2485, 1433).npy
loading saved_knn/attrsim_sampled_idx_(2485, 1433).npy
number of sampled: 21972
Epoch 0, training loss: 2.1042580604553223
Epoch 10, training loss: 1.226519227027893
Epoch 20, training loss: 0.5335997343063354
Epoch 30, training loss: 0.2053990364074707
Epoch 40, training loss: 0.10306744277477264
Epoch 50, training loss: 0.054293014109134674
Epoch 60, training loss: 0.03120998851954937
Epoch 70, training loss: 0.04105347767472267
Epoch 80, training loss: 0.031713783740997314
Epoch 90, training loss: 0.02100321464240551
Epoch 100, training loss: 0.026865530759096146
Epoch 110, training loss: 0.01693514734506607
Epoch 120, training loss: 0.0271962471306324
=== early stopping at 125, loss_val = 0.5914008021354675 ===
Test set results: loss= 0.6388 accuracy= 0.8048
0.8048289738430584


In [16]:
print((atk_acc - test_accuracy)*100)

-2.062374245472831


# Random

## GCN

In [172]:
# Setup Surrogate model
surrogate = GCN(nfeat=features.shape[1], nclass=labels.max().item()+1,
                nhid=64, dropout=0, with_relu=True, with_bias=True, device=device).to(device)
surrogate.fit(features, adj, labels, idx_train, idx_val, patience=100, verbose=True)

=== training gcn model ===
Epoch 0, training loss: 2.0254876613616943
Epoch 10, training loss: 0.16422221064567566
Epoch 20, training loss: 0.019253380596637726
Epoch 30, training loss: 0.0077318125404417515
Epoch 40, training loss: 0.00915240217000246
Epoch 50, training loss: 0.013718227855861187
Epoch 60, training loss: 0.016572128981351852
Epoch 70, training loss: 0.015872934833168983
Epoch 80, training loss: 0.014424710534512997
Epoch 90, training loss: 0.013599131256341934
Epoch 100, training loss: 0.013069466687738895
Epoch 110, training loss: 0.012576715089380741
=== early stopping at 113, loss_val = 0.4683011770248413 ===


In [173]:
preds=surrogate.predict(features,adj)
_, predicted_classes = torch.max(preds, 1)  # Returns values, indices; we need indices

# Step 3: Subset for test nodes
test_preds = predicted_classes[idx_test].cpu().numpy()
test_labels = labels[idx_test]

# Step 4: Calculate accuracy
correct = (test_preds == test_labels).sum().item()  # Count correct predictions
total = idx_test.shape[0]  # Total number of test nodes
test_accuracy = correct / total

print(f'Test Accuracy: {test_accuracy}')

Test Accuracy: 0.8395372233400402


In [174]:
# Setup Attack Model
model = Random(surrogate, nnodes=adj.shape[0], attack_structure=True, attack_features=False, device=device).to(device)
model.attack(adj, n_perturbations=budget)
modified_adj = model.modified_adj
# Convert SciPy sparse matrix to PyTorch tensor
modified_adj_torch = torch.tensor(modified_adj.toarray(), dtype=torch.float)

# Now you can safely call .cpu() if needed (useful if working with CUDA tensors)
modified_adj_torch = modified_adj_torch.cpu()

modified_adj = csr_matrix(modified_adj_torch)

In [175]:
atk_model = GCN(nfeat=features.shape[1], nclass=labels.max().item()+1,
                nhid=64, dropout=0, with_relu=True, with_bias=True, device=device).to(device)
atk_model.fit(features, modified_adj, labels, idx_train, idx_val, patience=100, verbose=True)

atk_acc = atk_model.test(idx_test)
print(atk_acc)

=== training gcn model ===
Epoch 0, training loss: 1.965752363204956
Epoch 10, training loss: 0.14442315697669983
Epoch 20, training loss: 0.01634359359741211
Epoch 30, training loss: 0.007796036545187235
Epoch 40, training loss: 0.010289414785802364
Epoch 50, training loss: 0.015795765444636345
Epoch 60, training loss: 0.01812082901597023
Epoch 70, training loss: 0.016661083325743675
Epoch 80, training loss: 0.015030885115265846
Epoch 90, training loss: 0.014180232770740986
Epoch 100, training loss: 0.013568446040153503
Epoch 110, training loss: 0.012998247519135475
=== early stopping at 112, loss_val = 0.465155690908432 ===
Test set results: loss= 0.5287 accuracy= 0.8290
0.8289738430583502


In [176]:
print((atk_acc - benchmark_clean)*100)

-1.8611670020120652


## GCNSVD

In [177]:
surrogate2 = GCNSVD(nfeat=features.shape[1],
          nhid=64,
          nclass=labels.max().item() + 1,
          dropout=0.5, device=device).to(device)
surrogate2.fit(features, adj, labels, idx_train, idx_val, k=50)

=== GCN-SVD: rank=50 ===
rank_after = 50
=== training gcn model ===
Epoch 0, training loss: 2.0072333812713623
Epoch 10, training loss: 0.4592343866825104
Epoch 20, training loss: 0.2288334220647812
Epoch 30, training loss: 0.1471269577741623
Epoch 40, training loss: 0.1256720870733261
Epoch 50, training loss: 0.10384183377027512
Epoch 60, training loss: 0.10801202058792114
Epoch 70, training loss: 0.10334079712629318
Epoch 80, training loss: 0.09238924086093903
Epoch 90, training loss: 0.08405647426843643
Epoch 100, training loss: 0.07731684297323227
Epoch 110, training loss: 0.07937982678413391
Epoch 120, training loss: 0.07834400981664658
Epoch 130, training loss: 0.07717201113700867
Epoch 140, training loss: 0.07578078657388687
Epoch 150, training loss: 0.07002373784780502
Epoch 160, training loss: 0.06499943137168884
Epoch 170, training loss: 0.06508328020572662
Epoch 180, training loss: 0.05942505970597267
Epoch 190, training loss: 0.05865178257226944
=== picking the best model a

In [178]:
preds=surrogate2.predict(features,adj)
_, predicted_classes = torch.max(preds, 1)  # Returns values, indices; we need indices

# Step 3: Subset for test nodes
test_preds = predicted_classes[idx_test].cpu().numpy()
test_labels = labels[idx_test]

# Step 4: Calculate accuracy
correct = (test_preds == test_labels).sum().item()  # Count correct predictions
total = idx_test.shape[0]  # Total number of test nodes
test_accuracy = correct / total

print(f'Test Accuracy: {test_accuracy}')

=== GCN-SVD: rank=50 ===
rank_after = 50
Test Accuracy: 0.7776659959758552


In [179]:
# Setup Attack Model
model = Random(surrogate2, nnodes=adj.shape[0], attack_structure=True, attack_features=False, device=device).to(device)
model.attack(adj, n_perturbations=budget)
modified_adj = model.modified_adj
# Convert SciPy sparse matrix to PyTorch tensor
modified_adj_torch = torch.tensor(modified_adj.toarray(), dtype=torch.float)

# Now you can safely call .cpu() if needed (useful if working with CUDA tensors)
modified_adj_torch = modified_adj_torch.cpu()

modified_adj = csr_matrix(modified_adj_torch)

In [180]:
atk_model = GCNSVD(nfeat=features.shape[1],
          nhid=64,
          nclass=labels.max().item() + 1,
          dropout=0.5, device=device).to(device)
atk_model.fit(features, modified_adj, labels, idx_train, idx_val, patience=100, verbose=True)

atk_acc = atk_model.test(idx_test)
print(atk_acc)

=== GCN-SVD: rank=50 ===
rank_after = 50
=== training gcn model ===
Epoch 0, training loss: 1.9951658248901367
Epoch 10, training loss: 0.4771680533885956
Epoch 20, training loss: 0.24277465045452118
Epoch 30, training loss: 0.1674921065568924
Epoch 40, training loss: 0.14359399676322937
Epoch 50, training loss: 0.10838336497545242
Epoch 60, training loss: 0.11508247256278992
Epoch 70, training loss: 0.10182808339595795
Epoch 80, training loss: 0.10541876405477524
Epoch 90, training loss: 0.09216145426034927
Epoch 100, training loss: 0.08674002438783646
Epoch 110, training loss: 0.08656434714794159
Epoch 120, training loss: 0.0795908272266388
Epoch 130, training loss: 0.07393693178892136
Epoch 140, training loss: 0.07294539362192154
Epoch 150, training loss: 0.07418721914291382
Epoch 160, training loss: 0.06426604837179184
Epoch 170, training loss: 0.06187780946493149
Epoch 180, training loss: 0.06268531829118729
Epoch 190, training loss: 0.06909627467393875
=== picking the best model 

In [181]:
print((atk_acc - benchmark_clean)*100)

-8.702213279678062


### GCN Jaccard

In [182]:
surrogate1 = GCNJaccard(nfeat=features.shape[1],
          nhid=64,
          nclass=labels.max().item() + 1,
          dropout=0.5, device=device).to(device)
surrogate1.fit(features, adj, labels, idx_train, idx_val, threshold=0.03)

removed 1015 edges in the original graph
=== training gcn model ===
Epoch 0, training loss: 1.9383213520050049
Epoch 10, training loss: 0.17206737399101257
Epoch 20, training loss: 0.04792378470301628
Epoch 30, training loss: 0.01681855134665966
Epoch 40, training loss: 0.013223609887063503
Epoch 50, training loss: 0.015077665448188782
Epoch 60, training loss: 0.020669620484113693
Epoch 70, training loss: 0.02149527333676815
Epoch 80, training loss: 0.02091405913233757
Epoch 90, training loss: 0.01609087735414505
Epoch 100, training loss: 0.01990603655576706
Epoch 110, training loss: 0.014299966394901276
Epoch 120, training loss: 0.014565463177859783
Epoch 130, training loss: 0.02086189202964306
Epoch 140, training loss: 0.013033526949584484
Epoch 150, training loss: 0.014002456329762936
Epoch 160, training loss: 0.013720656745135784
Epoch 170, training loss: 0.0118795745074749
Epoch 180, training loss: 0.013923942111432552
Epoch 190, training loss: 0.01620454154908657
=== picking the 

In [183]:
preds=surrogate1.predict(features,adj)
_, predicted_classes = torch.max(preds, 1)  # Returns values, indices; we need indices

# Step 3: Subset for test nodes
test_preds = predicted_classes[idx_test].cpu().numpy()
test_labels = labels[idx_test]

# Step 4: Calculate accuracy
correct = (test_preds == test_labels).sum().item()  # Count correct predictions
total = idx_test.shape[0]  # Total number of test nodes
test_accuracy = correct / total

print(f'Test Accuracy: {test_accuracy}')

removed 1015 edges in the original graph
Test Accuracy: 0.8003018108651911


In [184]:
# Setup Attack Model
model = Random(surrogate1, nnodes=adj.shape[0], attack_structure=True, attack_features=False, device=device).to(device)
model.attack(adj, n_perturbations=budget)

modified_adj = model.modified_adj
# Convert SciPy sparse matrix to PyTorch tensor
modified_adj_torch = torch.tensor(modified_adj.toarray(), dtype=torch.float)

# Now you can safely call .cpu() if needed (useful if working with CUDA tensors)
modified_adj_torch = modified_adj_torch.cpu()

modified_adj = csr_matrix(modified_adj_torch)

In [185]:
atk_model = GCNJaccard(nfeat=features.shape[1],
          nhid=64,
          nclass=labels.max().item() + 1,
          dropout=0.5, device=device).to(device)
atk_model.fit(features, modified_adj, labels, idx_train, idx_val, patience=100, verbose=True)

atk_acc = atk_model.test(idx_test)
print(atk_acc)

removed 647 edges in the original graph
=== training gcn model ===
Epoch 0, training loss: 2.0030083656311035
Epoch 10, training loss: 0.22502483427524567
Epoch 20, training loss: 0.04332933574914932
Epoch 30, training loss: 0.020152686163783073
Epoch 40, training loss: 0.025442497804760933
Epoch 50, training loss: 0.024266783148050308
Epoch 60, training loss: 0.01981409080326557
Epoch 70, training loss: 0.02382407709956169
Epoch 80, training loss: 0.020334556698799133
Epoch 90, training loss: 0.018671700730919838
Epoch 100, training loss: 0.01565883681178093
Epoch 110, training loss: 0.02035396173596382
Epoch 120, training loss: 0.01908489689230919
Epoch 130, training loss: 0.017074301838874817
Epoch 140, training loss: 0.018911724910140038
Epoch 150, training loss: 0.01562618277966976
Epoch 160, training loss: 0.0171397365629673
Epoch 170, training loss: 0.017664574086666107
Epoch 180, training loss: 0.015551199205219746
Epoch 190, training loss: 0.013927684165537357
=== picking the 

In [186]:
benchmark_clean

0.8475855130784709

In [187]:
print((atk_acc - benchmark_clean)*100)

-2.766599597585506


In [42]:
# Setup Surrogate model

features = normalize_feature(features)
surrogate = GCN(nfeat=features.shape[1], nclass=labels.max().item()+1,
                nhid=64, dropout=0, with_relu=True, with_bias=True, device=device).to(device)
surrogate.fit(features, adj, labels, idx_train, idx_val, patience=100, verbose=True)


preds=surrogate.predict(features,adj)
_, predicted_classes = torch.max(preds, 1)  # Returns values, indices; we need indices

# Step 3: Subset for test nodes
test_preds = predicted_classes[idx_test].cpu().numpy()
test_labels = labels[idx_test]

# Step 4: Calculate accuracy
correct = (test_preds == test_labels).sum().item()  # Count correct predictions
total = idx_test.shape[0]  # Total number of test nodes
test_accuracy = correct / total

print(f'Test Accuracy: {test_accuracy}')




NameError: name 'sp' is not defined

In [34]:
# Setup Attack Model
model1 = PGDAttack(surrogate, nnodes=adj.shape[0], attack_structure=True, attack_features=False, device=device).to(device)
# Attack
model1.attack(features, adj, labels, idx_train, n_perturbations=budget)
modified_adj = model1.modified_adj # modified_adj is a torch.tensor


  0%|          | 0/200 [00:00<?, ?it/s]


NotImplementedError: Could not run 'aten::fill_.Scalar' with arguments from the 'SparseCUDA' backend. This could be because the operator doesn't exist for this backend, or was omitted during the selective/custom build process (if using custom build). If you are a Facebook employee using PyTorch on mobile, please visit https://fburl.com/ptmfixes for possible resolutions. 'aten::fill_.Scalar' is only available for these backends: [CPU, CUDA, Meta, QuantizedCPU, QuantizedCUDA, SparseCsrCPU, SparseCsrCUDA, NestedTensorCPU, NestedTensorCUDA, BackendSelect, Python, FuncTorchDynamicLayerBackMode, Functionalize, Named, Conjugate, Negative, ZeroTensor, ADInplaceOrView, AutogradOther, AutogradCPU, AutogradCUDA, AutogradHIP, AutogradXLA, AutogradMPS, AutogradIPU, AutogradXPU, AutogradHPU, AutogradVE, AutogradLazy, AutogradMTIA, AutogradPrivateUse1, AutogradPrivateUse2, AutogradPrivateUse3, AutogradMeta, AutogradNestedTensor, Tracer, AutocastCPU, AutocastCUDA, FuncTorchBatched, BatchedNestedTensor